In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from pathlib import Path

Path("reports/charts").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

In [2]:
nav = pd.read_csv("data/raw/02_nav_history.csv")
fund = pd.read_csv("data/raw/01_fund_master.csv")
benchmark = pd.read_csv("data/raw/10_benchmark_indices.csv")

print(nav.columns)
print(fund.columns)
print(benchmark.columns)

Index(['amfi_code', 'date', 'nav'], dtype='object')
Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='object')
Index(['date', 'index_name', 'close_value'], dtype='object')


In [3]:
nav["date"] = pd.to_datetime(nav["date"])
nav = nav.sort_values(["amfi_code", "date"])

nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

nav.to_csv("data/processed/returns_computed.csv", index=False)

nav.head()

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210


In [4]:
def calculate_cagr(group):
    group = group.dropna().sort_values("date")
    nav_start = group["nav"].iloc[0]
    nav_end = group["nav"].iloc[-1]
    years = (group["date"].iloc[-1] - group["date"].iloc[0]).days / 365

    if years > 0:
        return (nav_end / nav_start) ** (1 / years) - 1
    return np.nan

cagr = nav.groupby("amfi_code").apply(calculate_cagr).reset_index()
cagr.columns = ["amfi_code", "cagr"]

cagr.to_csv("data/processed/cagr_report.csv", index=False)

cagr.head()

/var/folders/_5/x39y9r2522sgpmc7z5pqdbyh0000gn/T/ipykernel_9306/3296823381.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cagr = nav.groupby("amfi_code").apply(calculate_cagr).reset_index()


,amfi_code,cagr
0,100016,0.028788
1,100025,0.045425
2,100033,0.305184
3,101206,0.235044
4,101207,0.082066


In [5]:
risk_free_rate = 0.065

sharpe = nav.groupby("amfi_code")["daily_return"].agg(["mean", "std"]).reset_index()

sharpe["sharpe_ratio"] = (
    (sharpe["mean"] * 252 - risk_free_rate) /
    (sharpe["std"] * np.sqrt(252))
)

sharpe.to_csv("data/processed/sharpe_values.csv", index=False)

sharpe.head()

,amfi_code,mean,std,sharpe_ratio
0,100016,0.000142,0.009164,-0.201517
1,100025,0.000170,0.002460,-0.567095
2,100033,0.001080,0.011929,1.093699
3,101206,0.000852,0.009177,1.027213
4,101207,0.000424,0.016251,0.162661


In [6]:
def sortino_ratio(group):
    returns = group["daily_return"].dropna()
    downside = returns[returns < 0]

    if downside.std() == 0:
        return np.nan

    return ((returns.mean() * 252) - risk_free_rate) / (downside.std() * np.sqrt(252))

sortino = nav.groupby("amfi_code").apply(sortino_ratio).reset_index()
sortino.columns = ["amfi_code", "sortino_ratio"]

sortino.to_csv("data/processed/sortino_values.csv", index=False)

sortino.head()

/var/folders/_5/x39y9r2522sgpmc7z5pqdbyh0000gn/T/ipykernel_9306/1881490764.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sortino = nav.groupby("amfi_code").apply(sortino_ratio).reset_index()


,amfi_code,sortino_ratio
0,100016,-0.351047
1,100025,-0.941821
2,100033,1.829134
3,101206,1.799563
4,101207,0.276644


In [7]:
def max_drawdown(group):
    group = group.sort_values("date")
    running_max = group["nav"].cummax()
    drawdown = group["nav"] / running_max - 1
    return drawdown.min()

max_dd = nav.groupby("amfi_code").apply(max_drawdown).reset_index()
max_dd.columns = ["amfi_code", "max_drawdown"]

max_dd.to_csv("data/processed/max_drawdown.csv", index=False)

max_dd.head()

/var/folders/_5/x39y9r2522sgpmc7z5pqdbyh0000gn/T/ipykernel_9306/1373871472.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  max_dd = nav.groupby("amfi_code").apply(max_drawdown).reset_index()


,amfi_code,max_drawdown
0,100016,-0.247344
1,100025,-0.043083
2,100033,-0.162172
3,101206,-0.112916
4,101207,-0.354469


In [8]:
benchmark["date"] = pd.to_datetime(benchmark["date"])

print(benchmark.columns)
benchmark.head()

Index(['date', 'index_name', 'close_value'], dtype='object')


,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [13]:
print(benchmark.columns.tolist())
print(benchmark["index_name"].unique())

['date', 'index_name', 'close_value']
['NIFTY50' 'NIFTY100' 'NIFTY_MIDCAP150' 'BSE_SMALLCAP' 'NIFTY500'
 'CRISIL_LIQUID' 'CRISIL_GILT']


In [14]:
from scipy.stats import linregress

benchmark["date"] = pd.to_datetime(benchmark["date"])

# Select Nifty 100 data
nifty = benchmark[
    benchmark["index_name"] == "Nifty 100"
].copy()

nifty = nifty.sort_values("date")

# Benchmark returns
nifty["benchmark_return"] = nifty["close_value"].pct_change()

# Merge with fund NAV returns
merged = nav.merge(
    nifty[["date", "benchmark_return"]],
    on="date",
    how="inner"
)

results = []

for code, group in merged.groupby("amfi_code"):

    group = group.dropna(
        subset=["daily_return", "benchmark_return"]
    )

    if len(group) > 30:

        slope, intercept, r, p, std_err = linregress(
            group["benchmark_return"],
            group["daily_return"]
        )

        results.append({
            "amfi_code": code,
            "alpha": intercept * 252,
            "beta": slope
        })

alpha_beta = pd.DataFrame(results)

alpha_beta.to_csv(
    "data/processed/alpha_beta.csv",
    index=False
)

print(alpha_beta.head())

Empty DataFrame
Columns: []
Index: []


In [15]:
scorecard = cagr.merge(sharpe[["amfi_code", "sharpe_ratio"]], on="amfi_code", how="left")
scorecard = scorecard.merge(sortino, on="amfi_code", how="left")
scorecard = scorecard.merge(max_dd, on="amfi_code", how="left")
scorecard = scorecard.merge(alpha_beta, on="amfi_code", how="left")
scorecard = scorecard.merge(fund, on="amfi_code", how="left")

scorecard["cagr_rank"] = scorecard["cagr"].rank(pct=True)
scorecard["sharpe_rank"] = scorecard["sharpe_ratio"].rank(pct=True)
scorecard["alpha_rank"] = scorecard["alpha"].rank(pct=True)
scorecard["drawdown_rank"] = scorecard["max_drawdown"].rank(pct=True)

scorecard["fund_score"] = (
    scorecard["cagr_rank"] * 30 +
    scorecard["sharpe_rank"] * 25 +
    scorecard["alpha_rank"] * 20 +
    scorecard["drawdown_rank"] * 25
)

scorecard = scorecard.sort_values("fund_score", ascending=False)

scorecard.to_csv("data/processed/fund_scorecard.csv", index=False)

scorecard.head()

KeyError: 'amfi_code'